# Stage 5: Embeddings

Every chunk and every image record is embedded locally with **nomic-embed-text-v1.5** (ONNX on the CPU through
fastembed). Questions use the same model at query time. Code: `src/handbook_qa/embeddings.py` (decision 10).

Re-running: `embed_items` only embeds items whose text changed since the vectors in
`artifacts/embeddings/nomic-embed-text-v1.5.npz` were made, so a normal run takes seconds. Deleting that file
re-embeds everything (~20 minutes on this laptop; the first full run was done as a detached job, see the session
02 log). Batches are small and the file is saved after each one, so an interrupted run resumes.

In [1]:
import time

import numpy as np
import pandas as pd
import psutil

from handbook_qa.config import load_settings
from handbook_qa.embeddings import (
    DOCUMENT_PREFIX,
    MODEL_NAME,
    Embedder,
    VectorStore,
    embed_items,
    text_hash,
)
from handbook_qa.retrieval import load_corpus

settings = load_settings()
paths = settings.paths
print(MODEL_NAME, "| batch size", settings.embed.batch_size, "| threads", settings.embed.threads)
print(f"free RAM: {psutil.virtual_memory().available / 1e9:.1f} GB")

nomic-ai/nomic-embed-text-v1.5 | batch size 8 | threads 4
free RAM: 10.7 GB


## What gets embedded

Chunks (their full text, header included) and image records (their `retrieval_text`). Each text gets nomic's
task prefix: `search_document: ` for these, `search_query: ` for questions.

In [2]:
items = load_corpus(paths.chunks, paths.image_index)
kinds = pd.Series([i.kind for i in items]).value_counts()
print(kinds.to_string())
store = VectorStore.load(paths.vectors)
known = store.lookup()
stale = [i.id for i in items if known.get(i.id, ("", -1))[0] != text_hash(i.text)]
print(f"\nsaved vectors: {len(store.ids)} | items whose text changed or is new: {len(stale)}")
print(stale)

chunk    859
image     77

saved vectors: 936 | items whose text changed or is new: 8
['company/post-mortems/2025-10-21-feature-flags-recurring-outages#01', 'company/post-mortems/2025-10-21-feature-flags-recurring-outages#02', 'content/posthog-style-guide#03', 'content/posthog-style-guide#04', 'engineering/product-engineering#01', 'engineering/product-engineering#02', 'growth/use-case-selling/release-engineering#01', 'growth/use-case-selling/release-engineering#02']


## Bring the vectors up to date

Only the items above are embedded; everything else is reused.

In [3]:
process = psutil.Process()
started = time.perf_counter()
stats = embed_items([(i.id, i.text) for i in items], paths.vectors, settings.embed)
print(stats, f"in {time.perf_counter() - started:.1f}s")

store = VectorStore.load(paths.vectors)
norms = np.linalg.norm(store.vectors, axis=1)
print(f"store: {store.vectors.shape} {store.vectors.dtype}, norms {norms.min():.4f}-{norms.max():.4f}, "
      f"aligned with items: {store.ids == [i.id for i in items]}")

C:\Users\shaba\Documents\extra project\paradigm_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'embedded': 8, 'reused': 928} in 10.5s
store: (936, 768) float32, norms 1.0000-1.0000, aligned with items: True


## Checking the model setup

- fastembed mean-pools token vectors (what nomic specifies) but doesn't normalise; `Embedder` normalises, so cosine
  similarity is a dot product.
- Inputs are truncated at the model's 8,192-token limit. The longest chunk is well under it.
- A batch of 8 chunks, timed, with memory use.

In [4]:
embedder = Embedder(settings.embed)
tokenizer = embedder.model.model.tokenizer
lengths = [len(tokenizer.encode(DOCUMENT_PREFIX + i.text).ids) for i in items]
print(f"nomic tokens per item: median {int(np.median(lengths))}, max {max(lengths)} "
      f"(truncation at {tokenizer.truncation['max_length']})")

sample = [i.text for i in items[:8]]
started = time.perf_counter()
vectors = embedder.embed_documents(sample)
print(f"batch of 8: {time.perf_counter() - started:.1f}s, process memory {process.memory_info().rss / 1e9:.2f} GB")
print("re-embedding matches the stored vectors:", bool(np.allclose(vectors, store.matrix_for([i.id for i in items[:8]]),
                                                                   atol=1e-4)))

nomic tokens per item: median 529, max 893 (truncation at 8192)


batch of 8: 10.6s, process memory 1.90 GB
re-embedding matches the stored vectors: True


## Does it point the right way?

A few questions against chunks we know answer them, and against an unrelated chunk.

In [5]:
by_id = dict(zip(store.ids, store.vectors))
pairs = [
    ("Who should I ask for new artwork or merch?", "brand/art-requests#01"),
    ("How much credit do Y Combinator companies get?", "brand/startups#01"),
    ("How much RAM does the 4-core codespace have?", "img-p0355-x1727"),
]
unrelated = "engineering/product-design#01"
rows = []
for question, target in pairs:
    q = embedder.embed_query(question)
    scores = store.vectors @ q
    rows.append({"question": question, "target": target, "cos(target)": round(float(by_id[target] @ q), 3),
                 "cos(unrelated)": round(float(by_id[unrelated] @ q), 3),
                 "target rank among all": int((scores > by_id[target] @ q).sum()) + 1})
pd.DataFrame(rows)

,question,target,cos(target),cos(unrelated),target rank among all
0,Who should I ask for new artwork or merch?,brand/art-requests#01,0.789,0.563,1
1,How much credit do Y Combinator companies get?,brand/startups#01,0.750,0.495,2
2,How much RAM does the 4-core codespace have?,img-p0355-x1727,0.751,0.443,1
